<a href="https://colab.research.google.com/github/pjastr-uwm/fakultet_io_2026/blob/main/lab02/lab02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ćwiczenia 2: Przetwarzanie wstępne z NLTK i spaCy

## Przewodnik praktyczny

---

### Czym jest ten notatnik?

Ten notatnik to praktyczny przewodnik po **przetwarzaniu wstępnym tekstu** (preprocessing). Skupimy się na trzech kluczowych technikach:

1. **Stemming** — heurystyczne obcinanie końcówek słów
2. **Lematyzacja** — sprowadzanie do formy słownikowej z uwzględnieniem gramatyki
3. **Usuwanie stop words i normalizacja** — czyszczenie tekstu przed analizą

Będziemy pracować z dwoma bibliotekami: **NLTK** (podejście regułowe) i **spaCy** (modele językowe z wbudowaną lematyzacją i POS taggingiem).

**Struktura notatnika:**
- Sekcje z objaśnieniami i przykładami do uruchomienia
- Ćwiczenia do samodzielnego wykonania (komórki z komentarzem `# Wpisz swój kod poniżej`)
- Porównania NLTK vs spaCy na tych samych danych

---
## 1. Instalacja i konfiguracja środowiska

Instalujemy NLTK, spaCy oraz pobieramy modele językowe. W Google Colab wystarczy uruchomić poniższą komórkę raz na początku sesji.

In [77]:
# Instalacja pakietów
!pip install nltk spacy --quiet

# Pobranie modeli spaCy
!python -m spacy download en_core_web_sm --quiet
!python -m spacy download pl_core_news_sm --quiet

import nltk

# Pobranie zasobów NLTK
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)

print("Wszystko gotowe!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 90.2 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 57.4 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('pl_core_news_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Wszystko gotowe!


**Co zainstalowaliśmy?**

- `nltk` — biblioteka z regułowymi narzędziami NLP (stemmery, WordNet do lematyzacji, listy stop words)
- `spacy` — biblioteka z modelami językowymi, które wykonują lematyzację, POS tagging i inne zadania w jednym pipeline
- `en_core_web_sm` — mały model spaCy dla angielskiego
- `pl_core_news_sm` — mały model spaCy dla polskiego
- `wordnet` — baza leksykalna angielskiego, z której korzysta lematyzator NLTK
- `stopwords` — listy stop words NLTK dla różnych języków

---
## 2. Stemming w NLTK

### Czym jest stemming?

**Stemming** to heurystyczne obcinanie końcówek (sufiksów) słowa, aby uzyskać jego "rdzeń" (stem). Stemmer nie korzysta ze słownika — stosuje zestaw reguł obcinania.

Ważna cecha: wynik stemmingu **często nie jest poprawnym słowem** (np. `studies` → `studi`).

### Trzy popularne stemmery w NLTK

| Stemmer | Agresywność | Języki | Uwagi |
|---------|-------------|--------|-------|
| **Porter** | Umiarkowana | Angielski | Klasyczny (1980), najczęściej używany |
| **Lancaster** | Wysoka | Angielski | Bardziej agresywne obcinanie |
| **Snowball** | Umiarkowana | Wiele języków | Ulepszony Porter, obsługuje m.in. polski |

### 2.1 Porównanie stemmerów na tekście angielskim

In [78]:
from nltk.stem import PorterStemmer, LancasterStemmer, SnowballStemmer

# Inicjalizacja trzech stemmerów
porter = PorterStemmer()
lancaster = LancasterStemmer()
snowball = SnowballStemmer('english')

# Słowa testowe — formy regularne i nieregularne
test_words = [
    'running', 'runs', 'ran',
    'studies', 'studying', 'studied',
    'better', 'best', 'good',
    'easily', 'fairly', 'happily',
    'connection', 'connected', 'connecting',
    'university', 'universe',
]

print(f"{'Słowo':<16} {'Porter':<14} {'Lancaster':<14} {'Snowball':<14}")
print("-" * 58)
for word in test_words:
    p = porter.stem(word)
    l = lancaster.stem(word)
    s = snowball.stem(word)
    print(f"{word:<16} {p:<14} {l:<14} {s:<14}")

Słowo            Porter         Lancaster      Snowball      
----------------------------------------------------------
running          run            run            run           
runs             run            run            run           
ran              ran            ran            ran           
studies          studi          study          studi         
studying         studi          study          studi         
studied          studi          study          studi         
better           better         bet            better        
best             best           best           best          
good             good           good           good          
easily           easili         easy           easili        
fairly           fairli         fair           fair          
happily          happili        happy          happili       
connection       connect        connect        connect       
connected        connect        connect        connect       
connecting 

**Na co zwrócić uwagę:**

- `ran` — żaden stemmer nie łączy go z `run` (forma nieregularna, reguły obcinania nie pomagają)
- `better`, `best`, `good` — stemming nie rozumie stopniowania przymiotników
- `easily` → `easili` (Porter) — wynik nie jest poprawnym słowem
- `university` vs `universe` — mogą dać ten sam stem (over-stemming)

### 2.2 Over-stemming i under-stemming

Stemming popełnia dwa rodzaje błędów:

In [79]:
porter = PorterStemmer()

# Over-stemming: różne znaczenie, ale ten sam stem
print("OVER-STEMMING (zbyt agresywne obcinanie):")
print("="*50)
over_pairs = [
    ('university', 'universe'),
    ('organization', 'organ'),
    ('general', 'generate'),
]
for w1, w2 in over_pairs:
    s1, s2 = porter.stem(w1), porter.stem(w2)
    same = "TAK — błąd!" if s1 == s2 else "NIE"
    print(f"  {w1:<16} → {s1:<12} | {w2:<12} → {s2:<12} | Taki sam stem? {same}")

print()

# Under-stemming: to samo znaczenie, ale różne stemy
print("UNDER-STEMMING (zbyt mało obcięte):")
print("="*50)
under_pairs = [
    ('alumnus', 'alumni'),
    ('ran', 'running'),
    ('was', 'be'),
]
for w1, w2 in under_pairs:
    s1, s2 = porter.stem(w1), porter.stem(w2)
    same = "TAK" if s1 == s2 else "NIE — błąd!"
    print(f"  {w1:<16} → {s1:<12} | {w2:<12} → {s2:<12} | Taki sam stem? {same}")

OVER-STEMMING (zbyt agresywne obcinanie):
  university       → univers      | universe     → univers      | Taki sam stem? TAK — błąd!
  organization     → organ        | organ        → organ        | Taki sam stem? TAK — błąd!
  general          → gener        | generate     → gener        | Taki sam stem? TAK — błąd!

UNDER-STEMMING (zbyt mało obcięte):
  alumnus          → alumnu       | alumni       → alumni       | Taki sam stem? NIE — błąd!
  ran              → ran          | running      → run          | Taki sam stem? NIE — błąd!
  was              → wa           | be           → be           | Taki sam stem? NIE — błąd!


### Ćwiczenia do samodzielnego wykonania

**Ćwiczenie 2.1:** Zastosuj Porter Stemmer do poniższego zdania. Najpierw stokenizuj tekst za pomocą `word_tokenize()`, a potem zastosuj stemming do każdego tokenu. Wyświetl wynik w formie tabeli oryginał → stem.

```
The researchers were studying the effectiveness of different approaches to natural language processing.
```

In [80]:
# Miejsce na Ćwiczenie 2.1
# Wpisz swój kod poniżej:



**Ćwiczenie 2.2:** Porównaj wyniki Portera i Lancastera na słowach: `'generous'`, `'generation'`, `'generalize'`, `'generally'`, `'generic'`. Który stemmer lepiej rozróżnia te słowa?

In [81]:
# Miejsce na Ćwiczenie 2.2
# Wpisz swój kod poniżej:



---
## 3. Lematyzacja

### Czym się różni od stemmingu?

**Lematyzacja** sprowadza słowo do jego **formy słownikowej** (lematu). W przeciwieństwie do stemmingu:
- wynik jest zawsze **poprawnym słowem**
- uwzględnia **kontekst gramatyczny** (część mowy)
- korzysta ze **słownika lub modelu językowego**
- jest wolniejsza, ale dokładniejsza

Pokażemy dwa podejścia: **NLTK (WordNet)** i **spaCy (model językowy)**.

### 3.1 Lematyzacja w NLTK — WordNetLemmatizer

WordNetLemmatizer korzysta z bazy WordNet. Wymaga podania **części mowy** (`pos`), aby poprawnie lematyzować — domyślnie zakłada rzeczownik (`'n'`).

In [82]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

# Lematyzacja z podaniem części mowy (POS)
# 'n' = noun, 'v' = verb, 'a' = adjective, 'r' = adverb
test_cases = [
    ('studies', 'n'),    # rzeczownik
    ('studies', 'v'),    # czasownik
    ('better', 'a'),     # przymiotnik
    ('running', 'v'),    # czasownik
    ('mice', 'n'),       # rzeczownik (forma nieregularna)
    ('went', 'v'),       # czasownik (forma nieregularna)
    ('geese', 'n'),      # rzeczownik (forma nieregularna)
    ('easily', 'r'),     # przysłówek
]

pos_names = {'n': 'rzeczownik', 'v': 'czasownik', 'a': 'przymiotnik', 'r': 'przysłówek'}

print(f"{'Słowo':<14} {'POS':<14} {'Lemat':<14}")
print("-" * 42)
for word, pos in test_cases:
    lemma = lemmatizer.lemmatize(word, pos=pos)
    print(f"{word:<14} {pos_names[pos]:<14} {lemma:<14}")

Słowo          POS            Lemat         
------------------------------------------
studies        rzeczownik     study         
studies        czasownik      study         
better         przymiotnik    good          
running        czasownik      run           
mice           rzeczownik     mouse         
went           czasownik      go            
geese          rzeczownik     goose         
easily         przysłówek     easily        


**Uwaga:** Bez podania POS, lematyzator domyślnie traktuje słowo jako rzeczownik:

In [83]:
# Bez POS vs z POS — różnica w wynikach
print("Lematyzacja BEZ podania POS (domyślnie: noun):")
print("="*50)
words = ['running', 'better', 'studies', 'went', 'happier']
for w in words:
    no_pos = lemmatizer.lemmatize(w)          # domyślnie noun
    with_v = lemmatizer.lemmatize(w, pos='v')  # jako verb
    with_a = lemmatizer.lemmatize(w, pos='a')  # jako adj
    print(f"  {w:<12} bez POS: {no_pos:<12} jako verb: {with_v:<12} jako adj: {with_a:<12}")

print("\nWniosek: podanie poprawnej części mowy jest kluczowe!")

Lematyzacja BEZ podania POS (domyślnie: noun):
  running      bez POS: running      jako verb: run          jako adj: running     
  better       bez POS: better       jako verb: better       jako adj: good        
  studies      bez POS: study        jako verb: study        jako adj: studies     
  went         bez POS: went         jako verb: go           jako adj: went        
  happier      bez POS: happier      jako verb: happier      jako adj: happy       

Wniosek: podanie poprawnej części mowy jest kluczowe!


### 3.2 Lematyzacja w spaCy — podejście z modelem

spaCy wykonuje lematyzację automatycznie — model sam rozpoznaje część mowy i dobiera lemat. Nie trzeba ręcznie podawać POS.

In [84]:
import spacy

nlp_en = spacy.load('en_core_web_sm')

text = "The researchers were studying the effectiveness of different approaches to processing."
doc = nlp_en(text)

print(f"{'Token':<16} {'Lemat':<16} {'POS':<8} {'Szczegółowy POS':<16}")
print("-" * 56)
for token in doc:
    print(f"{token.text:<16} {token.lemma_:<16} {token.pos_:<8} {token.tag_:<16}")

Token            Lemat            POS      Szczegółowy POS 
--------------------------------------------------------
The              the              DET      DT              
researchers      researcher       NOUN     NNS             
were             be               AUX      VBD             
studying         study            VERB     VBG             
the              the              DET      DT              
effectiveness    effectiveness    NOUN     NN              
of               of               ADP      IN              
different        different        ADJ      JJ              
approaches       approach         NOUN     NNS             
to               to               ADP      IN              
processing       processing       NOUN     NN              
.                .                PUNCT    .               


**Zalety spaCy:**
- Automatyczne rozpoznawanie części mowy (POS)
- Nie trzeba ręcznie podawać POS jak w NLTK
- Poprawnie obsługuje formy nieregularne (`were` → `be`, `studying` → `study`)

### 3.3 Lematyzacja polskiego tekstu — spaCy

Dla polskiego NLTK nie oferuje lematyzacji — korzystamy ze spaCy z modelem `pl_core_news_sm`.

In [85]:
nlp_pl = spacy.load('pl_core_news_sm')

tekst = "Koty biegały po dachach starych kamienic i szukały ciepłych miejsc"
doc = nlp_pl(tekst)

print(f"{'Token':<16} {'Lemat':<16} {'POS':<8}")
print("-" * 40)
for token in doc:
    print(f"{token.text:<16} {token.lemma_:<16} {token.pos_:<8}")

Token            Lemat            POS     
----------------------------------------
Koty             Kot              NOUN    
biegały          biegać           VERB    
po               po               ADP     
dachach          dach             NOUN    
starych          stary            ADJ     
kamienic         kamienica        NOUN    
i                i                CCONJ   
szukały          szukać           VERB    
ciepłych         ciepły           ADJ     
miejsc           miejsce          NOUN    


**Sprawdźmy trudniejsze przypadki:**

In [86]:
# Formy fleksyjne tego samego słowa
zdania_pl = [
    "Widziałem piękne koty na dachach.",
    "Piękniejsze kociaki bawiły się na podwórku.",
    "Daliśmy kotom jedzenie wczoraj wieczorem.",
    "Najpiękniejszy kot spał na kanapie.",
]

nlp_pl = spacy.load('pl_core_news_sm')

for zdanie in zdania_pl:
    doc = nlp_pl(zdanie)
    lematy = [token.lemma_ for token in doc if token.pos_ not in ('PUNCT', 'SPACE')]
    print(f"Oryginał:  {zdanie}")
    print(f"Lematy:    {' | '.join(lematy)}")
    print()

Oryginał:  Widziałem piękne koty na dachach.
Lematy:    widzieć być | piękny | kot | na | dache

Oryginał:  Piękniejsze kociaki bawiły się na podwórku.
Lematy:    piękniejsz | kociak | bawić | się | na | podwórko

Oryginał:  Daliśmy kotom jedzenie wczoraj wieczorem.
Lematy:    Daliśmy | kot | jeść | wczoraj | wieczór

Oryginał:  Najpiękniejszy kot spał na kanapie.
Lematy:    piękny | kot | spać | na | kanapa



### 3.4 Porównanie: stemming vs lematyzacja — tabela zbiorcza

In [87]:
from nltk.stem import PorterStemmer, SnowballStemmer
from nltk.stem import WordNetLemmatizer
import spacy

porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()
nlp_en = spacy.load('en_core_web_sm')

test_words = ['better', 'running', 'geese', 'studies', 'was', 'easily', 'went', 'children']

print(f"{'Słowo':<12} {'Stemming':<12} {'NLTK lemat':<14} {'spaCy lemat':<14}")
print("-" * 52)
for word in test_words:
    stem = porter.stem(word)
    nltk_lemma = lemmatizer.lemmatize(word, pos='v')  # zakładamy verb
    spacy_lemma = nlp_en(word)[0].lemma_
    print(f"{word:<12} {stem:<12} {nltk_lemma:<14} {spacy_lemma:<14}")

print("\nUwaga: NLTK lemat z pos='v' — w praktyce trzeba znać POS wcześniej.")

Słowo        Stemming     NLTK lemat     spaCy lemat   
----------------------------------------------------
better       better       better         well          
running      run          run            run           
geese        gees         geese          geese         
studies      studi        study          study         
was          wa           be             be            
easily       easili       easily         easily        
went         went         go             go            
children     children     children       child         

Uwaga: NLTK lemat z pos='v' — w praktyce trzeba znać POS wcześniej.


### Ćwiczenia do samodzielnego wykonania

**Ćwiczenie 3.1:** Użyj `WordNetLemmatizer` z NLTK, aby zlematyzować poniższe słowa. Dla każdego słowa przetestuj lematyzację z dwoma różnymi wartościami `pos` (np. `'n'` i `'v'`). Które wyniki są poprawne?

Słowa: `'leaves'`, `'produced'`, `'worse'`, `'flying'`, `'feet'`

In [88]:
# Miejsce na Ćwiczenie 3.1
# Wpisz swój kod poniżej:



**Ćwiczenie 3.2:** Za pomocą spaCy (model `pl_core_news_sm`) zlematyzuj zdanie:

```
Programiści tworzyli zaawansowane aplikacje wykorzystujące sztuczną inteligencję do analizy danych medycznych.
```

Wyświetl wynik w formie tabeli: token, lemat, część mowy. Ile unikalnych lematów zawiera zdanie (bez interpunkcji)?

In [89]:
# Miejsce na Ćwiczenie 3.2
# Wpisz swój kod poniżej:



---
## 4. Usuwanie stop words

### Co to są stop words?

**Stop words** to bardzo częste słowa, które potencjalnie nie wnoszą informacji semantycznej. Usuwamy je, aby zmniejszyć szum w danych.

Pokażemy listy stop words z NLTK i spaCy, a także jak przygotować własną listę dla polskiego.

### 4.1 Stop words w NLTK — angielski

In [90]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Angielskie stop words z NLTK
stop_en = set(stopwords.words('english'))
print(f"Liczba angielskich stop words (NLTK): {len(stop_en)}")
print(f"Przykłady: {sorted(list(stop_en))[:20]}")

# Filtrowanie tekstu
sentence = "This is a very important sentence about natural language processing"
tokens = word_tokenize(sentence.lower())
filtered = [t for t in tokens if t not in stop_en]

print(f"\nPrzed filtrowaniem: {tokens}")
print(f"Po filtrowaniu:     {filtered}")
print(f"Usunięto {len(tokens) - len(filtered)} z {len(tokens)} tokenów")

Liczba angielskich stop words (NLTK): 198
Przykłady: ['a', 'about', 'above', 'after', 'again', 'against', 'ain', 'all', 'am', 'an', 'and', 'any', 'are', 'aren', "aren't", 'as', 'at', 'be', 'because', 'been']

Przed filtrowaniem: ['this', 'is', 'a', 'very', 'important', 'sentence', 'about', 'natural', 'language', 'processing']
Po filtrowaniu:     ['important', 'sentence', 'natural', 'language', 'processing']
Usunięto 5 z 10 tokenów


### 4.2 Stop words w spaCy

In [91]:
import spacy

nlp_en = spacy.load('en_core_web_sm')
nlp_pl = spacy.load('pl_core_news_sm')

# Stop words z modeli spaCy
print(f"Angielskie stop words (spaCy): {len(nlp_en.Defaults.stop_words)}")
print(f"Polskie stop words (spaCy):    {len(nlp_pl.Defaults.stop_words)}")

print(f"\nPrzykłady angielskie: {sorted(list(nlp_en.Defaults.stop_words))[:15]}")
print(f"Przykłady polskie:    {sorted(list(nlp_pl.Defaults.stop_words))[:15]}")

Angielskie stop words (spaCy): 326
Polskie stop words (spaCy):    383

Przykłady angielskie: ["'d", "'ll", "'m", "'re", "'s", "'ve", 'a', 'about', 'above', 'across', 'after', 'afterwards', 'again', 'against', 'all']
Przykłady polskie:    ['a', 'aby', 'ach', 'acz', 'aczkolwiek', 'aj', 'albo', 'ale', 'alez', 'ależ', 'ani', 'az', 'aż', 'bardziej', 'bardzo']


### 4.3 Polskie stop words — zewnętrzna lista

NLTK nie ma wbudowanych polskich stop words. Możemy pobrać listę z repozytorium `bieli/stopwords` lub użyć listy ze spaCy.

In [92]:
import urllib.request

# Pobranie polskich stop words z GitHub
url = "https://raw.githubusercontent.com/bieli/stopwords/master/polish.stopwords.txt"
response = urllib.request.urlopen(url)
stop_pl_bieli = set(response.read().decode("utf-8").splitlines())

print(f"Polskie stop words (bieli/stopwords): {len(stop_pl_bieli)}")
print(f"Przykłady: {sorted(list(stop_pl_bieli))[:20]}")

# Filtrowanie polskiego tekstu
zdanie = "To jest bardzo ważne zdanie o przetwarzaniu języka naturalnego"
tokeny = zdanie.lower().split()
filtrowane = [t for t in tokeny if t not in stop_pl_bieli]

print(f"\nPrzed: {tokeny}")
print(f"Po:    {filtrowane}")

Polskie stop words (bieli/stopwords): 350
Przykłady: ['a', 'aby', 'ach', 'acz', 'aczkolwiek', 'aj', 'albo', 'ale', 'alez', 'ależ', 'ani', 'az', 'aż', 'bardziej', 'bardzo', 'beda', 'bede', 'bedzie', 'bez', 'bo']

Przed: ['to', 'jest', 'bardzo', 'ważne', 'zdanie', 'o', 'przetwarzaniu', 'języka', 'naturalnego']
Po:    ['ważne', 'zdanie', 'przetwarzaniu', 'języka', 'naturalnego']


### 4.4 Filtrowanie stop words w spaCy — pipeline

spaCy oznacza stop words bezpośrednio na tokenach atrybutem `is_stop`:

In [93]:
nlp_pl = spacy.load('pl_core_news_sm')

tekst = "To jest bardzo ważne zdanie o przetwarzaniu języka naturalnego w nowoczesnych systemach"
doc = nlp_pl(tekst)

print(f"{'Token':<20} {'Stop word?':<12} {'POS':<8}")
print("-" * 40)
for token in doc:
    print(f"{token.text:<20} {'TAK' if token.is_stop else 'nie':<12} {token.pos_:<8}")

# Filtrowanie — zachowaj tylko tokeny, które nie są stop words i nie są interpunkcją
filtered = [token.text for token in doc if not token.is_stop and not token.is_punct]
print(f"\nPo filtrowaniu: {filtered}")

Token                Stop word?   POS     
----------------------------------------
To                   TAK          AUX     
jest                 TAK          AUX     
bardzo               TAK          ADV     
ważne                nie          ADJ     
zdanie               nie          NOUN    
o                    TAK          ADP     
przetwarzaniu        nie          NOUN    
języka               nie          NOUN    
naturalnego          nie          ADJ     
w                    TAK          ADP     
nowoczesnych         nie          ADJ     
systemach            nie          NOUN    

Po filtrowaniu: ['ważne', 'zdanie', 'przetwarzaniu', 'języka', 'naturalnego', 'nowoczesnych', 'systemach']


### 4.5 Kiedy usuwanie stop words szkodzi — negacja

Usuwanie stop words może **odwrócić sens** zdania, gdy negacja (`nie`, `not`) jest na liście.

In [94]:
# Problem z negacją
opinie = [
    "Film nie był dobry",
    "Film był dobry",
]

nlp_pl = spacy.load('pl_core_news_sm')

print("Usuwanie stop words z opinii:")
print("="*50)
for opinia in opinie:
    doc = nlp_pl(opinia)
    filtered = [t.text for t in doc if not t.is_stop and not t.is_punct]
    print(f"  Oryginał:    '{opinia}'")
    print(f"  Po filtr.:   {filtered}")
    print()

print("Obie opinie dają ten sam wynik — utrata negacji!")
print("Rozwiązanie: usuń 'nie' z listy stop words, gdy ważny jest sentyment.")

Usuwanie stop words z opinii:
  Oryginał:    'Film nie był dobry'
  Po filtr.:   ['Film', 'nie', 'dobry']

  Oryginał:    'Film był dobry'
  Po filtr.:   ['Film', 'dobry']

Obie opinie dają ten sam wynik — utrata negacji!
Rozwiązanie: usuń 'nie' z listy stop words, gdy ważny jest sentyment.


### 4.6 Dostosowywanie listy stop words

In [95]:
import spacy

nlp_pl = spacy.load('pl_core_news_sm')

# Sprawdźmy, czy 'nie' jest stop wordem w spaCy
print(f"'nie' jest stop wordem w spaCy PL? {nlp_pl.vocab['nie'].is_stop}")

# Usunięcie 'nie' z listy stop words (ważne dla analizy sentymentu)
nlp_pl.Defaults.stop_words.discard('nie')
nlp_pl.vocab['nie'].is_stop = False

# Dodanie nowych stop words specyficznych dla domeny
for word in ['np', 'itd', 'itp', 'tzw']:
    nlp_pl.Defaults.stop_words.add(word)
    nlp_pl.vocab[word].is_stop = True

# Test
print(f"\n'nie' jest stop wordem po zmianie? {nlp_pl.vocab['nie'].is_stop}")
print(f"'np' jest stop wordem po zmianie?  {nlp_pl.vocab['np'].is_stop}")

# Ponowne filtrowanie z poprawioną listą
doc = nlp_pl("Film nie był dobry")
filtered = [t.text for t in doc if not t.is_stop and not t.is_punct]
print(f"\n'Film nie był dobry' po filtr.: {filtered}")
print("Teraz 'nie' zostaje zachowane!")

'nie' jest stop wordem w spaCy PL? False

'nie' jest stop wordem po zmianie? False
'np' jest stop wordem po zmianie?  True

'Film nie był dobry' po filtr.: ['Film', 'nie', 'dobry']
Teraz 'nie' zostaje zachowane!


### Ćwiczenia do samodzielnego wykonania

**Ćwiczenie 4.1:** Porównaj listy angielskich stop words z NLTK i spaCy. Ile słów jest wspólnych? Ile jest w NLTK, ale nie w spaCy? Ile odwrotnie? Użyj operacji na zbiorach (`set`).


In [96]:
# Miejsce na Ćwiczenie 4.1
# Wpisz swój kod poniżej:



**Ćwiczenie 4.2:** Napisz funkcję `filter_with_exceptions(text, nlp, keep_words)`, która:
- tokenizuje tekst za pomocą spaCy
- usuwa stop words i interpunkcję
- ale zachowuje słowa z listy `keep_words` (nawet jeśli są stop wordami)

Przetestuj na zdaniach:
```
"To nie jest dobre rozwiązanie"
"Bardzo ważny projekt nie powinien być anulowany"
```
z `keep_words = ['nie', 'bardzo']`.

In [97]:
# Miejsce na Ćwiczenie 4.2
# Wpisz swój kod poniżej:



**Ćwiczenie 4.3:** Dla poniższego tekstu oblicz, jaki procent tokenów stanowią stop words (osobno dla NLTK i spaCy). Który zbiór stop words jest bardziej "agresywny"?

```
The development of artificial intelligence has changed the way we think about technology and its impact on our daily lives.
```

In [98]:
# Miejsce na Ćwiczenie 4.3
# Wpisz swój kod poniżej:



---
## 5. Normalizacja tekstu

### Co to jest normalizacja?

**Normalizacja** to sprowadzanie tekstu do ustandaryzowanej formy. Cel: redukcja wariantywności, która nie niesie informacji.

Główne techniki normalizacji:
- Zamiana na małe litery (lowercasing)
- Normalizacja Unicode (NFC/NFKC)
- Usuwanie znaków specjalnych i interpunkcji
- Normalizacja białych znaków
- Redukcja powtórzeń znaków

### 5.1 Lowercasing — kiedy tak, kiedy nie

In [99]:
# Lowercasing — proste, ale nie zawsze pożądane
tekst = "Apple wypuściło iPhone'a 15 Pro Max"
print(f"Oryginał:    {tekst}")
print(f"Lowercased:  {tekst.lower()}")
print()

# Problem: NER traci sygnał z wielkich liter
import spacy
nlp_en = spacy.load('en_core_web_sm')

for text in ["Reading is a city in England", "reading is a city in england"]:
    doc = nlp_en(text)
    ents = [(e.text, e.label_) for e in doc.ents]
    print(f"  '{text}'")
    print(f"  Rozpoznane encje: {ents}")
    print()

Oryginał:    Apple wypuściło iPhone'a 15 Pro Max
Lowercased:  apple wypuściło iphone'a 15 pro max

  'Reading is a city in England'
  Rozpoznane encje: [('England', 'GPE')]

  'reading is a city in england'
  Rozpoznane encje: [('england', 'GPE')]



### 5.2 Normalizacja Unicode

In [100]:
import unicodedata

# Problem: ten sam znak, dwie różne reprezentacje
s1 = "ą"               # ą jako pojedynczy znak (U+0105)
s2 = "a\u0328"         # a + combining ogonek (U+0061 + U+0328)

print(f"s1 = '{s1}', len = {len(s1)}")
print(f"s2 = '{s2}', len = {len(s2)}")
print(f"s1 == s2? {s1 == s2}")
print(f"Wizualnie identyczne? Tak!")
print()

# Rozwiązanie: normalizacja NFC
s2_nfc = unicodedata.normalize('NFC', s2)
print(f"Po NFC: s2_nfc = '{s2_nfc}', len = {len(s2_nfc)}")
print(f"s1 == s2_nfc? {s1 == s2_nfc}")
print()

# NFKC — normalizacja kompatybilna (ligatury, warianty znaków)
tekst_special = "½ ﬁrma ２０２４"
nfkc = unicodedata.normalize('NFKC', tekst_special)
print(f"Oryginał: '{tekst_special}'")
print(f"Po NFKC:  '{nfkc}'")

s1 = 'ą', len = 1
s2 = 'ą', len = 2
s1 == s2? False
Wizualnie identyczne? Tak!

Po NFC: s2_nfc = 'ą', len = 1
s1 == s2_nfc? True

Oryginał: '½ ﬁrma ２０２４'
Po NFKC:  '1⁄2 firma 2024'


### 5.3 Usuwanie znaków specjalnych i normalizacja białych znaków

In [101]:
import re

tekst = "Hej!!! :) Sprawdź tę stronę: https://example.com #nlp @user   ekstra  "
print(f"Oryginał: '{tekst}'")

# Krok po kroku:
# 1. Usuwanie URL-i
tekst_clean = re.sub(r'https?://\S+', '', tekst)
print(f"Bez URL:  '{tekst_clean}'")

# 2. Usuwanie wzmianek i hashtagów
tekst_clean = re.sub(r'[@#]\w+', '', tekst_clean)
print(f"Bez @/#:  '{tekst_clean}'")

# 3. Usuwanie interpunkcji (zachowujemy litery i cyfry)
tekst_clean = re.sub(r'[^\w\s]', '', tekst_clean)
print(f"Bez int.: '{tekst_clean}'")

# 4. Normalizacja białych znaków
tekst_clean = re.sub(r'\s+', ' ', tekst_clean).strip()
print(f"Finał:    '{tekst_clean}'")

Oryginał: 'Hej!!! :) Sprawdź tę stronę: https://example.com #nlp @user   ekstra  '
Bez URL:  'Hej!!! :) Sprawdź tę stronę:  #nlp @user   ekstra  '
Bez @/#:  'Hej!!! :) Sprawdź tę stronę:      ekstra  '
Bez int.: 'Hej  Sprawdź tę stronę      ekstra  '
Finał:    'Hej Sprawdź tę stronę ekstra'


### 5.4 Redukcja powtórzeń znaków

In [102]:
import re

# Typowe w mediach społecznościowych
examples = [
    "Suuuuuuper film!!!",
    "Nieeeeee!!!!!!",
    "Hahahahahaha",
    "Tak taaaaaak!!!",
]

for text in examples:
    # Redukcja: więcej niż 2 powtórzenia tego samego znaku → max 2
    reduced = re.sub(r'(.)\1{2,}', r'\1\1', text)
    print(f"  '{text}' → '{reduced}'")

  'Suuuuuuper film!!!' → 'Suuper film!!'
  'Nieeeeee!!!!!!' → 'Niee!!'
  'Hahahahahaha' → 'Hahahahahaha'
  'Tak taaaaaak!!!' → 'Tak taak!!'


### 5.5 Kompletny pipeline normalizacji

In [103]:
import unicodedata
import re

def normalize_text(text, lowercase=True, remove_urls=True,
                   remove_mentions=True, remove_special=True,
                   reduce_repeats=True):
    """Uniwersalny pipeline normalizacji tekstu."""

    # 1. Normalizacja Unicode (NFC)
    text = unicodedata.normalize('NFC', text)

    # 2. Usuwanie znaków kontrolnych i niewidocznych
    text = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)
    text = text.replace('\u200b', '')  # zero-width space

    # 3. Usuwanie URL-i
    if remove_urls:
        text = re.sub(r'https?://\S+', '', text)

    # 4. Usuwanie wzmianek i hashtagów
    if remove_mentions:
        text = re.sub(r'[@#]\w+', '', text)

    # 5. Redukcja powtórzeń
    if reduce_repeats:
        text = re.sub(r'(.)\1{2,}', r'\1\1', text)

    # 6. Opcjonalne lowercasing
    if lowercase:
        text = text.lower()

    # 7. Opcjonalne usuwanie znaków specjalnych
    if remove_special:
        text = re.sub(r'[^\w\s]', '', text)

    # 8. Normalizacja białych znaków
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Testy
test_texts = [
    "  Héllo!!!  \u200b Wörld  🌍  ",
    "Suuuper film!!! :) https://example.com #polecam @user",
    "Dr. Kowalski ma 3.5 mln zł. SUPER!!!",
    "a\u0328 vs ą — to samo?",
]

for text in test_texts:
    normalized = normalize_text(text)
    print(f"Przed: '{text}'")
    print(f"Po:    '{normalized}'")
    print()

Przed: '  Héllo!!!  ​ Wörld  🌍  '
Po:    'héllo wörld'

Przed: 'Suuuper film!!! :) https://example.com #polecam @user'
Po:    'suuper film'

Przed: 'Dr. Kowalski ma 3.5 mln zł. SUPER!!!'
Po:    'dr kowalski ma 35 mln zł super'

Przed: 'ą vs ą — to samo?'
Po:    'ą vs ą to samo'



### Ćwiczenia do samodzielnego wykonania

**Ćwiczenie 5.1:** Zmodyfikuj funkcję `normalize_text`, dodając parametr `remove_numbers=False`. Gdy jest `True`, funkcja powinna usuwać wszystkie cyfry z tekstu. Przetestuj na zdaniu:

```
"W 2024 roku firma zatrudniła 150 nowych pracowników w 3 oddziałach."
```

In [104]:
# Miejsce na Ćwiczenie 5.1
# Wpisz swój kod poniżej:



**Ćwiczenie 5.2:** Napisz funkcję `normalize_social_media(text)`, która:
- zamienia emotikony tekstowe `:)`, `:(`, `:D` na tagi `<POSITIVE>`, `<NEGATIVE>`, `<POSITIVE>`
- redukuje powtórzenia znaków (max 2)
- usuwa URL-e i wzmianki
- **nie** zmienia wielkości liter
- **nie** usuwa interpunkcji

Przetestuj na: `"Suuuuper film!!! :) :D Polecam!!! https://movie.com @everyone :("`

In [105]:
# Miejsce na Ćwiczenie 5.2
# Wpisz swój kod poniżej:



---
## 6. Kompletny pipeline — od surowego tekstu do czystych danych

Połączmy wszystkie techniki w jeden pipeline. Pokażemy dwa warianty: klasyczny (NLTK) i nowoczesny (spaCy).

### 6.1 Pipeline klasyczny — NLTK

In [106]:
import unicodedata
import re
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

def preprocess_nltk(text, language='english', use_stemming=True):
    """Klasyczny pipeline: normalizacja → tokenizacja → stop words → stemming."""

    # 1. Normalizacja
    text = unicodedata.normalize('NFC', text)
    text = text.lower()
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 2. Tokenizacja
    tokens = word_tokenize(text, language=language)

    # 3. Usuwanie stop words
    stop_words = set(stopwords.words(language))
    tokens = [t for t in tokens if t not in stop_words]

    # 4. Stemming (opcjonalny)
    if use_stemming:
        stemmer = PorterStemmer()
        tokens = [stemmer.stem(t) for t in tokens]

    return tokens

# Test
text = "The researchers have been studying the effectiveness of different NLP approaches. They found interesting results!"

print("Surowy tekst:")
print(f"  '{text}'")
print()

result_stem = preprocess_nltk(text, use_stemming=True)
result_nostem = preprocess_nltk(text, use_stemming=False)

print(f"Ze stemmingiem:  {result_stem}")
print(f"Bez stemmingu:   {result_nostem}")

Surowy tekst:
  'The researchers have been studying the effectiveness of different NLP approaches. They found interesting results!'

Ze stemmingiem:  ['research', 'studi', 'effect', 'differ', 'nlp', 'approach', 'found', 'interest', 'result']
Bez stemmingu:   ['researchers', 'studying', 'effectiveness', 'different', 'nlp', 'approaches', 'found', 'interesting', 'results']


### 6.2 Pipeline nowoczesny — spaCy

In [107]:
import spacy
import unicodedata
import re

def preprocess_spacy(text, nlp, use_lemma=True):
    """Nowoczesny pipeline: normalizacja → spaCy (tokenizacja + POS + lematyzacja) → filtrowanie."""

    # 1. Minimalna normalizacja (spaCy radzi sobie z resztą)
    text = unicodedata.normalize('NFC', text)
    text = re.sub(r'https?://\S+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    # 2. Przetwarzanie przez spaCy (tokenizacja + POS + lematyzacja w jednym kroku)
    doc = nlp(text)

    # 3. Filtrowanie: bez stop words, bez interpunkcji, bez spacji
    tokens = []
    for token in doc:
        if token.is_stop or token.is_punct or token.is_space:
            continue
        if use_lemma:
            tokens.append(token.lemma_.lower())
        else:
            tokens.append(token.text.lower())

    return tokens

# Test na angielskim
nlp_en = spacy.load('en_core_web_sm')
text = "The researchers have been studying the effectiveness of different NLP approaches. They found interesting results!"

print("Surowy tekst:")
print(f"  '{text}'")
print()

result_lemma = preprocess_spacy(text, nlp_en, use_lemma=True)
result_nolemma = preprocess_spacy(text, nlp_en, use_lemma=False)

print(f"Z lematyzacją:   {result_lemma}")
print(f"Bez lematyzacji: {result_nolemma}")

Surowy tekst:
  'The researchers have been studying the effectiveness of different NLP approaches. They found interesting results!'

Z lematyzacją:   ['researcher', 'study', 'effectiveness', 'different', 'nlp', 'approach', 'find', 'interesting', 'result']
Bez lematyzacji: ['researchers', 'studying', 'effectiveness', 'different', 'nlp', 'approaches', 'found', 'interesting', 'results']


### 6.3 Pipeline na polskim tekście

In [108]:
from nltk.stem import SnowballStemmer
from nltk.tokenize import word_tokenize # Dodane, aby word_tokenize było dostępne
import spacy # Dodane, aby spacy było dostępne

tekst_pl = """Naukowcy z Uniwersytetu Warszawskiego opublikowali nowe badania
dotyczące przetwarzania języka naturalnego. Wykorzystali zaawansowane
modele uczenia maszynowego do analizy polskich tekstów prasowych."""

print("Surowy tekst:")
print(f"  {tekst_pl.strip()}")
print()

# Pipeline spaCy (lematyzacja)
nlp_pl = spacy.load('pl_core_news_sm')
doc = nlp_pl(tekst_pl)
tokens_lemma = [t.lemma_.lower() for t in doc if not t.is_stop and not t.is_punct and not t.is_space]

# print(f"Stemming (Snowball PL):")
# print(f"  {tokens_stem}")
# print()
print(f"Lematyzacja (spaCy PL):")
print(f"  {tokens_lemma}")
print()
# print(f"Unikalne stemy:  {len(set(tokens_stem))}")
print(f"Unikalne lematy: {len(set(tokens_lemma))}")

Surowy tekst:
  Naukowcy z Uniwersytetu Warszawskiego opublikowali nowe badania
dotyczące przetwarzania języka naturalnego. Wykorzystali zaawansowane
modele uczenia maszynowego do analizy polskich tekstów prasowych.

Lematyzacja (spaCy PL):
  ['naukowiec', 'uniwersytet', 'warszawski', 'opublikować', 'nowy', 'badanie', 'dotyczyć', 'przetwarzać', 'język', 'naturalny', 'wykorzystać', 'zaawansowany', 'model', 'uczeć', 'maszynowy', 'analiza', 'polski', 'tekst', 'prasowy']

Unikalne lematy: 19


### Ćwiczenia do samodzielnego wykonania

**Ćwiczenie 6.1:** Przepuść poniższy tekst przez oba pipeline'y (`preprocess_nltk` i `preprocess_spacy`). Porównaj wyniki — ile tokenów daje każdy z pipeline'ów? Ile unikalnych tokenów?

```
Machine learning algorithms have been significantly improving over the last decade. Deep neural networks are now capable of understanding complex patterns in data, which was previously thought to be impossible.
```

In [109]:
# Miejsce na Ćwiczenie 6.1
# Wpisz swój kod poniżej:



**Ćwiczenie 6.2:** Napisz kompletny pipeline `preprocess_polish(text)`, który:
1. Normalizuje Unicode (NFC)
2. Usuwa URL-e
3. Przetwarza tekst przez spaCy (`pl_core_news_sm`)
4. Usuwa stop words, interpunkcję i liczby
5. Zwraca listę lematów w formie małych liter

Przetestuj na:
```
"W 2024 roku firma Google zaprezentowała 5 nowych modeli AI. Więcej na: https://ai.google"
```

In [110]:
# Miejsce na Ćwiczenie 6.2
# Wpisz swój kod poniżej:



**Ćwiczenie 6.3 (zaawansowane):** Dany jest zbiór 5 opinii o filmach. Dla każdej opinii:
1. Zastosuj pełny preprocessing (normalizacja + tokenizacja + usuwanie stop words + lematyzacja) za pomocą spaCy PL
2. Wyświetl oryginał i przetworzone tokeny
3. Oblicz, jaki procent oryginalnych tokenów został zachowany

Opinie:
```python
opinie = [
    "Ten film był absolutnie wspaniały! Polecam wszystkim.",
    "Nie podobał mi się ten film. Był nudny i zbyt długi.",
    "Aktorzy grali świetnie, ale fabuła mogłaby być lepsza.",
    "Najgorszy film, jaki kiedykolwiek widziałem!!!",
    "Film ok, nic specjalnego. Może warto obejrzeć na nudny wieczór.",
]
```

In [111]:
# Miejsce na Ćwiczenie 6.3
# Wpisz swój kod poniżej:

